<a href="https://colab.research.google.com/github/Harshilkumarghori/localrepo/blob/main/LLM_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Step 1: Collect Example Data

##Step 2: Preprocess Data (Python)

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt, welch
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv("measurement.csv")
time = df["time"].values
signal = df["signal"].values

# Filtering (low-pass)
b, a = butter(4, 0.2)
filtered = filtfilt(b, a, signal)

# Power spectrum
freqs, psd = welch(filtered, fs=1000)


##Step 3: Generate Structured Insights

In [ ]:
summary = {
    "mean_value": np.mean(filtered),
    "max_value": np.max(filtered),
    "dominant_freq": freqs[np.argmax(psd)],
    "anomalies": np.where(filtered > np.mean(filtered) + 3*np.std(filtered))[0].size
}


##Step 4: Use LLM to Create Report

In [ ]:
from openai import OpenAI
client = OpenAI()

prompt = f"""
You are an automotive test engineer. Summarize the following measurement data:
- Mean signal value: {summary['mean_value']:.2f}
- Max signal value: {summary['max_value']:.2f}
- Dominant frequency: {summary['dominant_freq']:.2f} Hz
- Number of anomalies detected: {summary['anomalies']}

Explain what these results could mean in the context of vibration analysis.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

report_text = response.choices[0].message["content"]
print(report_text)


##Step 5: Generate Visual + Written Report

In [ ]:
from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.multi_cell(0, 10, report_text)
pdf.image("signal_plot.png", x=10, y=100, w=180)
pdf.output("measurement_report.pdf")